In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [4]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    # model='gpt-5-nano',
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    # model='gpt-5-nano',
    model=model,
    tools=[square]
)

## Calling subagents

In [6]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    # model='gpt-5-nano',
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [7]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='a91529a1-481a-464f-8518-86c40826a1d6'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Calculating square root of 456**\n\nI need to find the square root of 456. I can either use a tool to compute it or calculate it myself. I can approximate that √456 is about 21.354. I might try linear interpolation, using the nearby squares of 21 and 22. I could also use the Newton-Raphson method for precision. Starting with x0 = 21.4 seems logical, and I\'ll adjust from there to refine my results.**Refining the square root calculation**\n\nI’m working on calculating the square root of 456 more precisely. Starting with an initial guess of x0 = 21.4, I need to compute 456 divided by 21.4 to find a better approximation. After some iterations, I find that x1 is around 21.363. I’ll check 21.363 squared to confirm how close it is to 456. I feel like I might 

In [9]:
question = "What is the square root of 22?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 22?', additional_kwargs={}, response_metadata={}, id='3faa857c-09b6-440f-a130-bd808972733a'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Calculating square root**\n\nI need to compute the square root of 22. I’ve got subagents that can help: one for calculating the square root and another for squaring. Since it’s just one computation, I’ll call the square root subagent to get sqrt(22). The approximate value is about 4.6904. I’ll use functions.call_subagent_1 with x set to 22 to get that result, then I’ll present it clearly for the user, including the exact expression and the approximate value.**Presenting results clearly**\n\nIn the final answer, I should present the information clearly. I'll say the square root of 22 is approximately 4.6904 when rounded to four decimal places. I’ll also include the exact expression, which is just sqrt(22). This way, the user has both the approximate value

In [11]:
question = "What is the square of 22?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square of 22?', additional_kwargs={}, response_metadata={}, id='b97cabd9-78e5-41a1-8dc1-edb868277f3a'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Calculating the square of 22**\n\nIn the final answer, I'll say that the square of 22 is 484. I could add a quick note about the calculation, like 22 * 22 = 484, just to clarify. But if the tool gives a different result, I’ll be flexible and adapt my response accordingly. It’s important to stay accurate! So, let's call the tool and get that confirmed.", 'reasoning_details': [{'summary': "**Calculating the square of 22**\n\nIn the final answer, I'll say that the square of 22 is 484. I could add a quick note about the calculation, like 22 * 22 = 484, just to clarify. But if the tool gives a different result, I’ll be flexible and adapt my response accordingly. It’s important to stay accurate! So, let's call the tool and get that confirmed.", 'type': 'reasoning.s